In [1]:
import tensorflow as tf
import numpy as np
import time
import os

In [2]:
# =========================================================
# 1. OBTENER Y HACER ETL AL DATASET
# =========================================================
print("--- Paso 1: Obtención y ETL del Dataset ---")
# Usaremos MNIST (imágenes de dígitos 28x28) por su rapidez de entrenamiento.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Limpieza y Estandarización: Normalizamos los píxeles (de 0-255 a 0.0-1.0)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape: Agregamos la dimensión del canal requerida por las capas Conv2D (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

--- Paso 1: Obtención y ETL del Dataset ---
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [3]:
# =========================================================
# 2. ENTRENAR EL MODELO BASE (Float32)
# =========================================================
print("\n--- Paso 2: Entrenando el Modelo Base ---")
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Entrenamos el modelo original en formato FP32
model.fit(x_train, y_train, epochs=3, batch_size=64, validation_split=0.1)

# Guardamos el modelo base
model.save('modelo_base.h5')


--- Paso 2: Entrenando el Modelo Base ---
Epoch 1/3


C:\Users\Alonso\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9229 - loss: 0.2693 - val_accuracy: 0.9698 - val_loss: 0.1101
Epoch 2/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9694 - loss: 0.1018 - val_accuracy: 0.9790 - val_loss: 0.0738
Epoch 3/3
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9808 - loss: 0.0651 - val_accuracy: 0.9832 - val_loss: 0.0629


In [4]:
# =========================================================
# 3. CUANTIZACIÓN DEL MODELO (PTQ - INT8)
# =========================================================
print("\n--- Paso 3: Cuantizando el Modelo a INT8 ---")

# Función generadora para la "Calibración".
# Provee muestras reales para que el cuantizador defina los rangos de los tensores.
def representative_dataset():
    for data in tf.data.Dataset.from_tensor_slices(x_train).batch(1).take(200):
        yield [data]

# Configuramos el convertidor de TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset

# Forzamos a que las operaciones de entrada/salida e internas sean INT8
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8  
converter.inference_output_type = tf.int8 

# Realizamos la conversión
tflite_quant_model = converter.convert()

with open('modelo_cuantizado.tflite', 'wb') as f:
    f.write(tflite_quant_model)


--- Paso 3: Cuantizando el Modelo a INT8 ---
INFO:tensorflow:Assets written to: C:\Users\Alonso\AppData\Local\Temp\tmpt4jz52kc\assets


INFO:tensorflow:Assets written to: C:\Users\Alonso\AppData\Local\Temp\tmpt4jz52kc\assets


Saved artifact at 'C:\Users\Alonso\AppData\Local\Temp\tmpt4jz52kc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2635407029264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635407030224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635407029840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635407030608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635407029456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2635407029072: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\Alonso\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [5]:
# =========================================================
# 4. ANÁLISIS INGENIERIL Y COMPARACIÓN DE MÉTRICAS
# =========================================================
print("\n--- Paso 4: Comparación y Métricas ---")

# A. Consumo de Memoria (Footprint / Size)
size_base_kb = os.path.getsize('modelo_base.h5') / 1024
size_quant_kb = os.path.getsize('modelo_cuantizado.tflite') / 1024
print(f"Tamaño Modelo Base (FP32): {size_base_kb:.2f} KB")
print(f"Tamaño Modelo Cuantizado (INT8): {size_quant_kb:.2f} KB")
print(f"--> Reducción de almacenamiento: {(1 - size_quant_kb/size_base_kb)*100:.2f}%")

# B. Evaluación de Precisión y Latencia
def evaluate_tflite_model(tflite_model_path, x_test, y_test):
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    
    scale, zero_point = input_details['quantization']
    
    correct_predictions = 0
    start_time = time.time()
    
    for i in range(len(x_test)):
        # Estandarizar la entrada manualmente a los parámetros del modelo cuantizado (INT8)
        input_data = x_test[i:i+1]
        if scale != 0:
            input_data = (input_data / scale + zero_point).astype(input_details['dtype'])
            
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details['index'])
        if np.argmax(output_data) == y_test[i]:
            correct_predictions += 1
            
    end_time = time.time()
    accuracy = correct_predictions / len(x_test)
    avg_latency_ms = ((end_time - start_time) / len(x_test)) * 1000
    
    return accuracy, avg_latency_ms


--- Paso 4: Comparación y Métricas ---
Tamaño Modelo Base (FP32): 2071.73 KB
Tamaño Modelo Cuantizado (INT8): 176.12 KB
--> Reducción de almacenamiento: 91.50%


In [6]:
# Resultados Base
loss, base_acc = model.evaluate(x_test, y_test, verbose=0)
start_base = time.time()
model.predict(x_test[:1000], verbose=0)
base_latency_ms = ((time.time() - start_base) / 1000) * 1000

# Resultados Cuantizado
quant_acc, quant_latency_ms = evaluate_tflite_model('modelo_cuantizado.tflite', x_test, y_test)

print(f"\nPrecisión Modelo Base: {base_acc*100:.2f}%")
print(f"Precisión Modelo Cuantizado: {quant_acc*100:.2f}%")
print(f"--> Caída de precisión: {(base_acc - quant_acc)*100:.2f} puntos porcentuales")

print(f"\nLatencia Inferencia Base (CPU Python): {base_latency_ms:.2f} ms")
print(f"Latencia Inferencia Cuantizado (TFLite Interpreter): {quant_latency_ms:.2f} ms")

C:\Users\Alonso\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Precisión Modelo Base: 97.72%
Precisión Modelo Cuantizado: 97.73%
--> Caída de precisión: -0.01 puntos porcentuales

Latencia Inferencia Base (CPU Python): 0.25 ms
Latencia Inferencia Cuantizado (TFLite Interpreter): 0.09 ms
